# 01 — Dataset Understanding (Phase 3)

**Project:** Cross-border payments product case study (independent PM portfolio project).
**Source:** The World Bank, Remittance Prices Worldwide, available at http://remittanceprices.worldbank.org

This notebook documents *what the raw workbook contains* and *how reliable its structure is*. It is a
dataset-understanding step only:

* it does **not** clean, merge, or transform the data;
* it does **not** write anything to `data/raw/` (the workbook is opened read-only, and its SHA-256 is checked
  before and after the run);
* it does **not** produce product findings or recommendations. Counts and distributions shown here describe
  the *structure* of the file, not the remittance market.

Outputs: machine-generated structural tables in `outputs/tables/`. Written interpretation lives in
`docs/data_dictionary.md`, `docs/schema_comparison.md` and `docs/dataset_quality_report.md`.

**Run:** from the repository root, `jupyter nbconvert --to notebook --execute --inplace notebooks/01_dataset_understanding.ipynb`
(takes a few minutes; the post-2016 sheet XML is ~300 MB uncompressed).

## 0. Setup and raw-file integrity check
The expected hash is the one recorded in `docs/dataset_notes.md` when the file was added in Phase 2.

In [1]:
from pathlib import Path
import hashlib, html, re, zipfile
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from itertools import combinations

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 120)

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = REPO / "data" / "raw" / "rpw_dataset_2011_2025_q3.xlsx"
TABLES = REPO / "outputs" / "tables"
TABLES.mkdir(parents=True, exist_ok=True)
EXPECTED_SHA256 = "f1d7265b1856d61812c158d3d03fae686dbff58735deabe016a6bfe23482d2a8"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

hash_before = sha256(RAW)
print(RAW.relative_to(REPO), RAW.stat().st_size, "bytes")
print("sha256:", hash_before)
assert hash_before == EXPECTED_SHA256, "Raw workbook differs from the Phase 2 file"

data/raw/rpw_dataset_2011_2025_q3.xlsx 50780339 bytes
sha256: f1d7265b1856d61812c158d3d03fae686dbff58735deabe016a6bfe23482d2a8


## 1. Workbook package structure
An `.xlsx` file is a ZIP of XML parts. Reading the parts directly (read-only) shows things pandas hides:
declared sheet dimensions, text boxes (drawings), hyperlinks, formulas and external links.

In [2]:
NS = {
    "m": "http://schemas.openxmlformats.org/spreadsheetml/2006/main",
    "r": "http://schemas.openxmlformats.org/officeDocument/2006/relationships",
    "rel": "http://schemas.openxmlformats.org/package/2006/relationships",
    "a": "http://schemas.openxmlformats.org/drawingml/2006/main",
}
zf = zipfile.ZipFile(RAW, "r")  # read-only handle

def rels(part):
    folder, name = part.rsplit("/", 1)
    path = f"{folder}/_rels/{name}.rels"
    if path not in zf.namelist():
        return {}
    root = ET.fromstring(zf.read(path))
    return {r.get("Id"): (r.get("Type").rsplit("/", 1)[-1], r.get("Target"), r.get("TargetMode"))
            for r in root.findall("rel:Relationship", NS)}

wb_root = ET.fromstring(zf.read("xl/workbook.xml"))
wb_rels = rels("xl/workbook.xml")
sheet_parts = {}
for s in wb_root.find("m:sheets", NS):
    rid = s.get(f"{{{NS['r']}}}id")
    sheet_parts[s.get("name")] = "xl/" + wb_rels[rid][1]

def declared_dimension(part):
    with zf.open(part) as f:
        head = f.read(4096).decode("utf-8", "ignore")
    m = re.search(r'<dimension ref="([^"]+)"', head)
    return m.group(1) if m else None

structure = pd.DataFrame([
    {"sheet": name, "xml_part": part,
     "uncompressed_bytes": zf.getinfo(part).file_size,
     "declared_dimension": declared_dimension(part),
     "relationships": ", ".join(sorted({t for t, _, _ in rels(part).values()}))}
    for name, part in sheet_parts.items()])
structure

,sheet,xml_part,uncompressed_bytes,declared_dimension,relationships
0,Terms of Use,xl/worksheets/sheet1.xml,1670,B28:B31,"drawing, hyperlink, printerSettings"
1,Methodology,xl/worksheets/sheet2.xml,1046,A1,"drawing, printerSettings"
2,Legend,xl/worksheets/sheet3.xml,4820,A1:B36,"printerSettings, table"
3,Countries,xl/worksheets/sheet4.xml,54088,A1:I217,printerSettings
4,Dataset (up to Q1 2016),xl/worksheets/sheet5.xml,69806053,A1:AO49492,printerSettings
5,Dataset (from Q2 2016),xl/worksheets/sheet6.xml,298455536,A1:BV204470,printerSettings


In [3]:
print("Other workbook-level relationships:")
for rid, (typ, target, mode) in wb_rels.items():
    if typ != "worksheet":
        print(f"  {typ:15s} {target}")
print("\nDefined names (auto-filter ranges):")
for d in wb_root.iter(f"{{{NS['m']}}}definedName"):
    print(" ", d.get("name"), d.text)

Other workbook-level relationships:
  theme           theme/theme1.xml
  externalLink    externalLinks/externalLink1.xml
  calcChain       calcChain.xml
  sharedStrings   sharedStrings.xml
  styles          styles.xml

Defined names (auto-filter ranges):
  _xlnm._FilterDatabase Countries!$A$2:$F$217
  _xlnm._FilterDatabase 'Dataset (from Q2 2016)'!$A$1:$AP$204470
  _xlnm._FilterDatabase 'Dataset (up to Q1 2016)'!$A$1:$AO$49492


### 1.1 Terms of Use and Methodology sheets
Both sheets store their content in **drawing text boxes**, not cells. `pandas.read_excel` therefore returns
an empty (Methodology) or nearly empty (Terms of Use: only the four "REFERENCES" link cells) frame.
We extract the text from the drawing XML.

In [4]:
def drawing_text(sheet):
    part = sheet_parts[sheet]
    out = []
    for typ, target, _ in rels(part).values():
        if typ == "drawing":
            dpart = "xl/" + target.replace("../", "")
            root = ET.fromstring(zf.read(dpart))
            for p in root.iter(f"{{{NS['a']}}}p"):
                t = "".join(x.text or "" for x in p.iter(f"{{{NS['a']}}}t")).strip()
                if t:
                    out.append(t)
    return out

for sheet in ["Terms of Use", "Methodology"]:
    print("=" * 30, sheet, "=" * 30)
    for para in drawing_text(sheet):
        print(para, "\n")

print("Cell content of 'Terms of Use' (via pandas):")
print(pd.read_excel(RAW, sheet_name="Terms of Use", header=None).dropna(how="all").to_string())
print("\nHyperlinks on 'Terms of Use':")
for rid, (typ, target, mode) in rels(sheet_parts["Terms of Use"]).items():
    if typ == "hyperlink":
        print(" ", target)

============================== Terms of Use ==============================
Terms of Use Summary 

The use of the data is subject to the following conditions. 

You are free to copy, distribute, adapt, display or include the data in other products for commercial and noncommercial purposes at no cost subject to certain limitations summarized below. 

You must include attribution for the data you use as follows: "The World Bank, Remittance Prices Worldwide, available at http://remittanceprices.worldbank.org". 

You must not claim or imply that The World Bank endorses your use of the data by or use The World Bank’s logo(s) or trademark(s) in conjunction with such use. 

Other parties may have ownership interests in some of the materials contained on The World Bank Web site. For example, we maintain a list of some specific data within the Datasets that you may not redistribute or reuse without first contacting the original content provider, as well as information regarding how to contact th

     0                                1
27 NaN                      REFERENCES:
28 NaN   Terms of use: Restricted Data 
29 NaN        Terms of use for datasets
30 NaN  World Bank Terms and Conditions

Hyperlinks on 'Terms of Use':
  http://go.worldbank.org/R6942GMMH0
  http://go.worldbank.org/OJC02YMLA0
  http://go.worldbank.org/C09SUA7BK0


Observations (structural): the Methodology sheet describes how the World Bank computes **averages**
(Global Average, Weighted Average, MTO Index, SmaRT) from the data. It does **not** define row-level formulas
for fee, FX margin or total cost; those definitions come only from the Legend (next section). Non-transparent
RSPs are excluded from the World Bank's Global Average.

## 2. Legend sheet
The Legend is the only in-workbook field dictionary. It uses a generic `cc(X)` prefix for the two surveyed amounts and lists the Q2 2016 changes.

In [5]:
legend = pd.read_excel(RAW, sheet_name="Legend", header=None, names=["term", "definition"])
legend.to_csv(TABLES / "legend_raw.csv", index=False)
legend

,term,definition
0,Legend,NaN
1,id,id number applied to each service
2,period,data collection period
3,source,country where money is sent from
4,destination,country where money is sent to
5,_code,ISO 3166-1 alpha-3 country code
6,_name,country name
7,_region,country's World Bank region
8,_income,country's income group
9,_lending,country's lending category


## 3. Countries sheet
A reference table of country codes and classifications. Row 1 is a title/blank row; the header is row 2.

In [6]:
countries_raw = pd.read_excel(RAW, sheet_name="Countries", header=None)
print(countries_raw.head(3).to_string())
countries = pd.read_excel(RAW, sheet_name="Countries", header=1)
print(countries.shape)
display(countries.head())
print("\nValue counts per classification column (including the '..' placeholder):")
for c in countries.columns[2:]:
    print(f"\n{c}:", countries[c].value_counts(dropna=False).to_dict())
print("\nDuplicate codes:", countries.iloc[:, 0].duplicated().sum())

                                                                                                                                   0             1                                                  2             3                 4       5
0  World Bank Country and Lending Groups\n(updated Q3 2025 for info see: http://data.worldbank.org/about/country-and-lending-groups)           NaN                                                NaN           NaN               NaN   Other
1                                                                                                    ISO 3166-1 alpha-3 country code  Country name                                             Region  Income Group  Lending category  G8/G20
2                                                                                                                                AFG   Afghanistan  Middle East, North Africa, Afghanistan & Pakistan    Low income               IDA      ..


(215, 6)


,ISO 3166-1 alpha-3 country code,Country name,Region,Income Group,Lending category,G8/G20
0,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income,IDA,..
1,ALB,Albania,Europe & Central Asia,Upper middle income,IBRD,..
2,DZA,Algeria,"Middle East, North Africa, Afghanistan & Pakistan",Upper middle income,IBRD,..
3,ASM,American Samoa,East Asia & Pacific,High income,..,..
4,AND,Andorra,..,High income,..,..



Value counts per classification column (including the '..' placeholder):

Region: {'..': 76, 'Sub-Saharan Africa': 47, 'Latin America & Caribbean': 26, 'East Asia & Pacific': 24, 'Europe & Central Asia': 21, 'Middle East, North Africa, Afghanistan & Pakistan': 15, 'South Asia': 6}

Income Group: {'High income': 83, 'Upper middle income': 54, 'Lower middle income': 49, 'Low income': 27, '..': 2}

Lending category: {'..': 71, 'IBRD': 66, 'IDA': 60, 'Blend': 18}

G8/G20: {'..': 196, 'G20': 11, 'G8/G20': 8}

Duplicate codes: 0


## 4. Load the two historical data sheets (unchanged, in memory only)
`dtype=object` keeps the raw Python value types (str / int / float / datetime) produced by openpyxl so that
type inconsistencies inside a column stay visible. For formula cells pandas/openpyxl return the **cached value**
stored in the file (verified in Section 6). Nothing is written back.

In [7]:
PRE, POST = "Dataset (up to Q1 2016)", "Dataset (from Q2 2016)"
# keep_default_na=False: pandas would otherwise silently turn literal strings such as "N/A" or "NA" into NaN.
raw_frames = pd.read_excel(RAW, sheet_name=[PRE, POST], header=0, dtype=object, keep_default_na=False, na_values=[])
for name, df in raw_frames.items():
    lit = {c: int(df[c].isin(["N/A", "NA", "n/a", "None", "null", "NULL", "#N/A"]).sum()) for c in df.columns}
    print(name, "literal NA-like strings that default pandas settings would hide:", {k: v for k, v in lit.items() if v})
# With these settings empty cells and empty-string cells both arrive as "". For profiling only, treat "" as blank.
data = {name: df.replace({"": np.nan}) for name, df in raw_frames.items()}
pre, post = data[PRE], data[POST]
for name, df in data.items():
    print(f"{name}: {df.shape[0]:,} data rows x {df.shape[1]} columns read by pandas")
print("\nPost-2016 columns without a header:", [c for c in post.columns if str(c).startswith("Unnamed")])

Dataset (up to Q1 2016) literal NA-like strings that default pandas settings would hide: {'product': 214, 'speed actual': 10, 'coverage': 2}
Dataset (from Q2 2016) literal NA-like strings that default pandas settings would hide: {}


/tmp/ipykernel_10120/2910969260.py:8: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data = {name: df.replace({"": np.nan}) for name, df in raw_frames.items()}


Dataset (up to Q1 2016): 49,491 data rows x 41 columns read by pandas
Dataset (from Q2 2016): 204,469 data rows x 47 columns read by pandas

Post-2016 columns without a header: ['Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46']


## 5. Column inventory and schema comparison (by header name)

In [8]:
pre_cols, post_cols = list(pre.columns), [c for c in post.columns if not str(c).startswith("Unnamed")]
all_cols = list(dict.fromkeys(pre_cols + post_cols))
schema = pd.DataFrame({
    "column": all_cols,
    "pre_position": [pre_cols.index(c) if c in pre_cols else None for c in all_cols],
    "post_position": [post_cols.index(c) if c in post_cols else None for c in all_cols],
})
schema["in_pre"] = schema.pre_position.notna()
schema["in_post"] = schema.post_position.notna()
schema["status"] = np.select([schema.in_pre & schema.in_post, schema.in_pre], ["both", "pre only"], "post only")
schema.to_csv(TABLES / "schema_header_comparison.csv", index=False)
print(schema.status.value_counts().to_dict())
schema[schema.status != "both"]

{'both': 36, 'post only': 6, 'pre only': 5}


,column,pre_position,post_position,in_pre,in_post,status
16,product,16.0,NaN,True,False,pre only
17,sending location,17.0,NaN,True,False,pre only
35,note1,35.0,NaN,True,False,pre only
37,coverage,37.0,NaN,True,False,pre only
38,pick-up method,38.0,NaN,True,False,pre only
41,payment instrument,NaN,16.0,False,True,post only
42,access point,NaN,17.0,False,True,post only
43,Standard Note,NaN,35.0,False,True,post only
44,receiving network coverage,NaN,37.0,False,True,post only
45,pickup location,NaN,38.0,False,True,post only


In [9]:
legend_terms = set(legend.term.dropna().str.strip().str.lower())
print("Legend mentions 'sending network coverage':", "sending network coverage" in legend_terms)
print("Any post-2016 header containing 'sending network':",
      [c for c in post.columns if "sending network" in str(c).lower()])
print("Any post-2016 header containing 'coverage':", [c for c in post.columns if "coverage" in str(c).lower()])

Legend mentions 'sending network coverage': True
Any post-2016 header containing 'sending network': []
Any post-2016 header containing 'coverage': ['receiving network coverage']


## 6. Post-2016 sheet: populated columns, unlabeled columns, formulas and cached values
We stream the worksheet XML once and record, per column letter: non-empty cells, formula cells, formula cells
with a cached `<v>` value, whether formulas reference an external workbook (`[1]`), and example formulas.

In [10]:
def col_letter(ref):
    return re.match(r"[A-Z]+", ref).group(0)

def col_index(letters):
    n = 0
    for ch in letters:
        n = n * 26 + ord(ch) - 64
    return n - 1

M = f"{{{NS['m']}}}"
cells_total, cells_value = Counter(), Counter()
f_count, f_cached, f_external, f_shared_master = Counter(), Counter(), Counter(), Counter()
f_examples = defaultdict(list)
formula_cells = {}          # (row, col) -> cached value as stored in XML
with zf.open(sheet_parts[POST]) as fh:
    for event, el in ET.iterparse(fh, events=("end",)):
        if el.tag == M + "c":
            ref = el.get("r"); col = col_letter(ref)
            cells_total[col] += 1
            v = el.find(M + "v"); isel = el.find(M + "is"); f = el.find(M + "f")
            if (v is not None and v.text not in (None, "")) or isel is not None:
                cells_value[col] += 1
            if f is not None:
                f_count[col] += 1
                if v is not None and v.text not in (None, ""):
                    f_cached[col] += 1
                if f.text and "[1]" in f.text:
                    f_external[col] += 1
                if f.text:
                    f_shared_master[col] += 1
                    if len(f_examples[col]) < 3:
                        f_examples[col].append(f"{ref}: ={f.text}")
                row = int(ref[len(col):])
                formula_cells[(row, col)] = (v.text if v is not None else None, el.get("t"))
            el.clear()
        elif el.tag == M + "row":
            el.clear()

letters = sorted(cells_total, key=col_index)
header_by_letter = {}
for i, c in enumerate(post.columns):
    L = ""; n = i + 1
    while n:
        n, r = divmod(n - 1, 26); L = chr(65 + r) + L
    header_by_letter[L] = None if str(c).startswith("Unnamed") else c
xml_cols = pd.DataFrame([{
    "column_letter": L, "header": header_by_letter.get(L),
    "cells_in_xml": cells_total[L], "cells_with_value": cells_value[L] - (1 if header_by_letter.get(L) else 0),
    "formula_cells": f_count[L], "formula_cells_with_cached_value": f_cached[L],
    "formulas_referencing_external_workbook": f_external[L],
    "formula_examples": " | ".join(f_examples[L])} for L in letters])
xml_cols.to_csv(TABLES / "post2016_xml_column_scan.csv", index=False)
print("Declared dimension:", declared_dimension(sheet_parts[POST]))
print("Column letters that appear in the XML at all:", len(letters), f"({letters[0]}..{letters[-1]})")
xml_cols

Declared dimension: A1:BV204470
Column letters that appear in the XML at all: 53 (A..BV)


,column_letter,header,cells_in_xml,cells_with_value,formula_cells,formula_cells_with_cached_value,formulas_referencing_external_workbook,formula_examples
0,A,id,204470,204469,0,0,0,
1,B,period,204470,204469,0,0,0,
2,C,source_code,204470,204469,0,0,0,
3,D,source_name,204470,204469,0,0,0,
4,E,source_region,204470,204469,0,0,0,
5,F,source_income,204470,204469,0,0,0,
6,G,source_lending,204470,204469,0,0,0,
7,H,source_G8G20,204470,204469,0,0,0,
8,I,destination_code,204470,204469,0,0,0,
9,J,destination_name,204470,204469,37,37,37,"J133066: =VLOOKUP(I133066,[1]Countries!A$3:F$217,2,FALSE) | J133143: =VLOOKUP(I133143,[1]Countries!A$3:F$217,2,FALSE..."


In [11]:
print("External link parts and targets:")
for name in zf.namelist():
    if name.startswith("xl/externalLinks/") and name.endswith(".xml"):
        ext_root = ET.fromstring(zf.read(name))
        print(" ", name, "sheet names in linked workbook:",
              [s.get("val") for s in ext_root.iter(M + "sheetName")])
        for typ, target, mode in rels(name).values():
            print("   ", typ, mode, target)

External link parts and targets:
  xl/externalLinks/externalLink1.xml sheet names in linked workbook: ['Terms of Use', 'Methodology', 'Legend', 'Countriesold', 'Dataset (up to Q1 2016)', 'Dataset (from Q2 2016)', 'Countries', 'CountriesQ320', 'CountriesQ321', 'CountriesQ322', 'HistoricalCountryCat']
    externalLinkPath External ../../../../../../Users/schen/Documents/WeChat%20Files/yanningchen/FileStorage/File/2024-07/rpw_dataset_2011_2023_q1_with%20formula.xlsx
    externalLinkPath External /Users/schen/Documents/WeChat%20Files/yanningchen/FileStorage/File/2024-07/rpw_dataset_2011_2023_q1_with%20formula.xlsx


**Are cached formula values what pandas returns?** For every formula cell we compare the cached XML value
with the value pandas loaded for that cell. We also check whether the cached values agree with what the formula
*would* produce from other columns in the same file (corridor = source_code & destination_code; J/K = lookups
into a Countries table in the external workbook), without recalculating or modifying anything in the workbook.

In [12]:
mismatch = 0; checked = 0
for (row, col), (v, t) in formula_cells.items():
    val = post.iat[row - 2, col_index(col)]
    checked += 1
    if str(val) != str(v) and not (t is None and v is not None and float(v) == float(val)):
        mismatch += 1
print(f"formula cells checked: {checked:,}; pandas value != cached XML value: {mismatch}")

ap_rows = [r for (r, c) in formula_cells if c == "AP"]
ap = post.iloc[[r - 2 for r in ap_rows]]
print("AP formula rows: periods", ap.period.unique().tolist())
print("AP cached corridor == source_code+destination_code:",
      (ap.corridor == ap.source_code + ap.destination_code).mean())

jk = [(r, c) for (r, c) in formula_cells if c in ("J", "K")]
jk_df = pd.DataFrame([{"row": r, "col": c, "cached": post.iat[r - 2, col_index(c)],
                       "period": post.at[r - 2, "period"], "destination_code": post.at[r - 2, "destination_code"]}
                      for r, c in jk])
lookup_name = dict(zip(countries.iloc[:, 0], countries.iloc[:, 1]))
lookup_region = dict(zip(countries.iloc[:, 0], countries.iloc[:, 2]))
jk_df["value_in_this_workbook_Countries"] = [
    (lookup_name if c == "J" else lookup_region).get(code) for c, code in zip(jk_df.col, jk_df.destination_code)]
print(jk_df.groupby(["col", "period", "destination_code", "cached", "value_in_this_workbook_Countries"],
                    dropna=False).size())

formula cells checked: 6,690; pandas value != cached XML value: 0
AP formula rows: periods ['2024_4Q']
AP cached corridor == source_code+destination_code: 1.0
col  period   destination_code  cached                 value_in_this_workbook_Countries
J    2023_1Q  KSV               Kosovo                 Kosovo                              37
K    2023_1Q  KSV               Europe & Central Asia  Europe & Central Asia                2
dtype: int64


### 6.1 Unlabeled post-2016 columns (pandas `Unnamed: 42`–`Unnamed: 46`)

In [13]:
unnamed = [c for c in post.columns if str(c).startswith("Unnamed")]
nn = post[unnamed].notna()
print("non-null counts:", nn.sum().to_dict())
print("periods where any unlabeled column is populated:", post.loc[nn.any(axis=1), "period"].value_counts().to_dict())
sub = post.loc[nn.any(axis=1)]
print(sub[["period", "source_code", "destination_code"] + unnamed].drop_duplicates().head(8).to_string())
print("\nPattern checks on populated rows:")
print("  col42 values:", sub[unnamed[0]].astype(str).unique()[:5])
print("  col43 == source_code + '2025':", (sub[unnamed[1]] == sub.source_code + "2025").mean())
print("  col44 == destination_code + '2025':", (sub[unnamed[2]] == sub.destination_code + "2025").mean())
print("  col45 == source_code + '2026':", (sub[unnamed[3]] == sub.source_code + "2026").mean())
print("  col46 == destination_code + '2026':", (sub[unnamed[4]] == sub.destination_code + "2026").mean())

non-null counts: {'Unnamed: 42': 6470, 'Unnamed: 43': 6470, 'Unnamed: 44': 6470, 'Unnamed: 45': 6470, 'Unnamed: 46': 6470}
periods where any unlabeled column is populated: {'2025_3Q': 6470}
         period source_code destination_code  Unnamed: 42 Unnamed: 43 Unnamed: 44 Unnamed: 45 Unnamed: 46
197999  2025_3Q         AGO              NAM       2025.0     AGO2025     NAM2025     AGO2026     NAM2026
198000  2025_3Q         ARE              SSD       2025.0     ARE2025     SSD2025     ARE2026     SSD2026
198001  2025_3Q         ARE              IND       2025.0     ARE2025     IND2025     ARE2026     IND2026
198005  2025_3Q         ARE              EGY       2025.0     ARE2025     EGY2025     ARE2026     EGY2026
198007  2025_3Q         ARE              LKA       2025.0     ARE2025     LKA2025     ARE2026     LKA2026
198008  2025_3Q         ARE              NPL       2025.0     ARE2025     NPL2025     ARE2026     NPL2026
198009  2025_3Q         ARE              PAK       2025.0     ARE202

These columns contain only derived concatenation keys (a constant year and code+year strings), appear only in
2025_3Q rows and have no header or Legend entry. They look like helper/working columns, not survey fields.
Their purpose is **not documented**; they carry no information beyond `source_code`, `destination_code` and a year.

## 7. Value-type consistency inside columns
Excel lets a column mix numbers, text and dates. We count Python value types per column and per period.

In [14]:
def type_name(v):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "missing"
    return type(v).__name__

type_rows = []
for sheet, df in [(PRE, pre), (POST, post)]:
    for c in df.columns:
        t = df[c].map(type_name)
        by = pd.crosstab(df.period, t)
        present = [x for x in by.columns if x != "missing"]
        if len(present) > 1:
            for typ in present:
                periods = by.index[by[typ] > 0].tolist()
                type_rows.append({"sheet": sheet, "column": c, "python_type": typ, "rows": int(by[typ].sum()),
                                  "n_periods": len(periods),
                                  "periods": ", ".join(periods) if len(periods) <= 6 else f"{periods[0]} .. {periods[-1]}"})
mixed_types = pd.DataFrame(type_rows)
mixed_types.to_csv(TABLES / "mixed_value_types.csv", index=False)
mixed_types

,sheet,column,python_type,rows,n_periods,periods
0,Dataset (from Q2 2016),id,int,191181,36,2016_2Q .. 2025_3Q
1,Dataset (from Q2 2016),id,str,13288,3,"2020_2Q, 2024_3Q, 2024_4Q"
2,Dataset (from Q2 2016),cc1 lcu amount,int,191178,35,2016_2Q .. 2025_3Q
3,Dataset (from Q2 2016),cc1 lcu amount,str,13291,2,"2024_3Q, 2024_4Q"
4,Dataset (from Q2 2016),cc1 denomination amount,int,191178,35,2016_2Q .. 2025_3Q
5,Dataset (from Q2 2016),cc1 denomination amount,str,13291,2,"2024_3Q, 2024_4Q"
6,Dataset (from Q2 2016),cc1 lcu fee,float,85272,36,2016_2Q .. 2025_3Q
7,Dataset (from Q2 2016),cc1 lcu fee,int,105890,35,2016_2Q .. 2025_3Q
8,Dataset (from Q2 2016),cc1 lcu fee,str,13285,2,"2024_3Q, 2024_4Q"
9,Dataset (from Q2 2016),cc1 lcu fx rate,float,160756,36,2016_2Q .. 2025_3Q


## 8. Placeholder and missing-value conventions
We count true blanks, the `..` token, and other placeholder-like strings, per column.

In [15]:
PLACEHOLDERS = ["..", "", "n/a", "N/A", "NA", "-", "Not available", "none", "None", "0"]
ph_rows = []
for sheet, df in [(PRE, pre), (POST, post)]:
    for c in df.columns:
        s = df[c]
        rec = {"sheet": sheet, "column": c, "rows": len(s), "blank_or_null": int(s.isna().sum())}
        st = s[s.map(lambda v: isinstance(v, str))].astype(str)
        rec["whitespace_only"] = int((st.str.strip() == "").sum())
        for p in PLACEHOLDERS[:-1]:
            if p:
                rec[f"'{p}'"] = int((st.str.strip() == p).sum())
        ph_rows.append(rec)
placeholders = pd.DataFrame(ph_rows)
placeholders.to_csv(TABLES / "placeholder_counts.csv", index=False)
cols = [c for c in placeholders.columns if c.startswith("'") or c == "whitespace_only"]
placeholders[(placeholders[cols].sum(axis=1) > 0)]

,sheet,column,rows,blank_or_null,whitespace_only,'..','n/a','N/A','NA','-','Not available','none','None'
4,Dataset (up to Q1 2016),source_region,49491,0,0,45452,0,0,0,0,0,0,0
6,Dataset (up to Q1 2016),source_lending,49491,0,0,42712,0,0,0,0,0,0,0
7,Dataset (up to Q1 2016),source_G8G20,49491,0,0,13748,0,0,0,0,0,0,0
10,Dataset (up to Q1 2016),destination_region,49491,0,0,1656,0,0,0,0,0,0,0
11,Dataset (up to Q1 2016),destination_income,49491,0,0,121,0,0,0,0,0,0,0
12,Dataset (up to Q1 2016),destination_lending,49491,0,0,784,0,0,0,0,0,0,0
13,Dataset (up to Q1 2016),destination_G8G20,49491,0,0,39216,0,0,0,0,0,0,0
16,Dataset (up to Q1 2016),product,49491,0,0,0,0,214,0,0,0,0,0
17,Dataset (up to Q1 2016),sending location,49491,0,0,0,0,0,0,0,16559,0,0
18,Dataset (up to Q1 2016),speed actual,49491,0,0,0,0,10,0,0,0,0,0


In [16]:
print("Which countries carry '..' in each classification column (distinct codes):")
for sheet, df in [(PRE, pre), (POST, post)]:
    for side in ["source", "destination"]:
        for k in ["region", "income", "lending", "G8G20"]:
            c = f"{side}_{k}"
            codes = sorted(df.loc[df[c] == "..", f"{side}_code"].unique())
            if codes:
                inc = df.loc[df[c] == "..", f"{side}_income"].value_counts().to_dict()
                print(f"{sheet[:24]:24s} {c:20s} n_codes={len(codes):3d} income of those rows={inc}")

Which countries carry '..' in each classification column (distinct codes):
Dataset (up to Q1 2016)  source_region        n_codes= 25 income of those rows={'High income: OECD': 38722, 'High income: nonOECD': 6454, 'Upper middle income': 276}
Dataset (up to Q1 2016)  source_lending       n_codes= 22 income of those rows={'High income: OECD': 37784, 'High income: nonOECD': 4928}
Dataset (up to Q1 2016)  source_G8G20         n_codes= 22 income of those rows={'High income: OECD': 7445, 'High income: nonOECD': 3992, 'Upper middle income': 1220, 'Low income': 978, 'Lower middle income': 113}
Dataset (up to Q1 2016)  destination_region   n_codes=  7 income of those rows={'High income: nonOECD': 819, 'High income: OECD': 716, '..': 121}
Dataset (up to Q1 2016)  destination_income   n_codes=  1 income of those rows={'..': 121}
Dataset (up to Q1 2016)  destination_lending  n_codes=  5 income of those rows={'High income: nonOECD': 471, '..': 121, 'Upper middle income': 104, 'High income: OECD': 88

Structural reading of `..`: in `*_G8G20` it marks "not a G8/G20 member" (the Countries sheet uses the same
token for 196 of 215 countries); in `*_region` and `*_lending` it co-occurs with high-income countries (and a
few others), consistent with those countries not being assigned a developing-region / lending category in the
classification used. The workbook does **not** define `..` explicitly — this is an inference from co-occurrence
and must be treated as such.

## 9. Categorical label variants (case / whitespace)
Labels that differ only by case or surrounding whitespace.

In [17]:
CATS = ["period", "source_code", "destination_code", "firm", "firm_type", "product", "sending location",
        "payment instrument", "access point", "speed actual", "transparent", "coverage", "receiving network coverage",
        "pick-up method", "pickup method", "pickup location", "cc1 lcu code", "source_income", "destination_income",
        "source_region", "destination_region"]
var_rows = []
for sheet, df in [(PRE, pre), (POST, post)]:
    for c in CATS:
        if c not in df:
            continue
        s = df[c].dropna().astype(str)
        g = s.groupby(s.str.strip().str.lower()).agg(lambda x: sorted(x.unique()))
        for key, variants in g.items():
            if len(variants) > 1:
                var_rows.append({"sheet": sheet, "column": c, "variants": " | ".join(variants),
                                 "rows": int(s.str.strip().str.lower().eq(key).sum())})
label_variants = pd.DataFrame(var_rows)
label_variants.to_csv(TABLES / "label_variants.csv", index=False)
label_variants

,sheet,column,variants,rows
0,Dataset (up to Q1 2016),firm,Azimo | azimo,485
1,Dataset (up to Q1 2016),pick-up method,"ATM, Bank Account | ATM, Bank account",426
2,Dataset (up to Q1 2016),pick-up method,Bank Account | Bank account | bank Account,13220
3,Dataset (up to Q1 2016),pick-up method,"Bank Account, Home Delivery | Bank account, Home delivery",67
4,Dataset (up to Q1 2016),pick-up method,"Cash, Bank Account | Cash, Bank account",1035
5,Dataset (up to Q1 2016),pick-up method,"Cash, Bank Account, Home Delivery | Cash, Bank account, Home delivery",22
6,Dataset (up to Q1 2016),pick-up method,"Door to Door, Bank Account | Door to door, Bank account",5
7,Dataset (up to Q1 2016),pick-up method,Home Delivery | Home delivery,936
8,Dataset (up to Q1 2016),pick-up method,Own/Partner bank account | Own/partner Bank Account | own/partner Bank Account,1471
9,Dataset (from Q2 2016),firm,ACE Money Transfer | Ace Money Transfer,447


In [18]:
print("Codes mapped to more than one name (renames over time):")
for sheet, df in [(PRE, pre), (POST, post)]:
    for side in ["source", "destination"]:
        g = df.groupby(f"{side}_code")[f"{side}_name"].unique()
        for code_, names in g[g.map(len) > 1].items():
            print(f"  {sheet[:24]:24s} {side:11s} {code_}: {list(names)}")
print("\nCorridor vs source_code+destination_code:")
for sheet, df in [(PRE, pre), (POST, post)]:
    bad = df[df.corridor != df.source_code + df.destination_code]
    print(f"  {sheet}: {len(bad)} mismatches", bad.groupby(["period", "corridor", "source_code", "destination_code"]).size().to_dict())

Codes mapped to more than one name (renames over time):
  Dataset (from Q2 2016)   source      CZE: ['Czech Republic', 'Czechia']
  Dataset (from Q2 2016)   source      TUR: ['Turkey', 'Türkiye']
  Dataset (from Q2 2016)   destination MKD: ['Macedonia, FYR', 'North Macedonia']
  Dataset (from Q2 2016)   destination SWZ: ['Swaziland', 'Eswatini']
  Dataset (from Q2 2016)   destination TUR: ['Turkey', 'Türkiye']

Corridor vs source_code+destination_code:
  Dataset (up to Q1 2016): 0 mismatches {}
  Dataset (from Q2 2016): 95 mismatches {('2021_4Q', 'AUTXKX', 'AUT', 'KSV'): 9, ('2021_4Q', 'CHEXKX', 'CHE', 'KSV'): 9, ('2021_4Q', 'DEUXKX', 'DEU', 'KSV'): 8, ('2021_4Q', 'ITAXKX', 'ITA', 'KSV'): 8, ('2023_1Q', 'AUTXKX', 'AUT', 'KSV'): 23, ('2023_1Q', 'CHEXKX', 'CHE', 'KSV'): 13, ('2023_1Q', 'DEUXKX', 'DEU', 'KSV'): 13, ('2023_1Q', 'ITAXKX', 'ITA', 'KSV'): 12}


## 10. Structural profile per sheet
Pure profiling; numeric statistics are computed on `pd.to_numeric(errors="coerce")` copies held in memory
(text-stored numbers such as `"200.00"` are counted as numeric here, and are flagged in Section 7).
"Missing" below means an empty cell or an empty string; literal strings such as `N/A` are *not* counted as missing.

In [19]:
NUMERIC = ["id", "cc1 lcu amount", "cc1 denomination amount", "cc1 lcu fee", "cc1 lcu fx rate", "cc1 fx margin",
           "cc1 total cost %", "cc2 lcu amount", "cc2 denomination amount", "cc2 lcu fee", "cc2 lcu fx rate",
           "cc2 fx margin", "cc2 total cost %", "inter lcu bank fx"]

def profile(df, sheet):
    meaningful = [c for c in df.columns if not str(c).startswith("Unnamed") and df[c].notna().any()]
    periods = sorted(df.period.unique())
    summary = {
        "sheet": sheet, "rows": len(df), "columns_read": df.shape[1], "meaningful_columns": len(meaningful),
        "first_period": periods[0], "last_period": periods[-1], "n_periods": len(periods),
        "unique_source_countries": df.source_code.nunique(), "unique_destination_countries": df.destination_code.nunique(),
        "unique_corridors": df.corridor.nunique(), "unique_source_dest_pairs": df.groupby(["source_code", "destination_code"]).ngroups,
        "unique_firms": df.firm.nunique(), "unique_firm_types": df.firm_type.nunique(),
        "unique_ids": df.id.astype(str).nunique(),
        "duplicate_rows_full": int(df.astype(str).duplicated().sum()),
        "duplicate_rows_excluding_id": int(df.drop(columns="id").astype(str).duplicated().sum()),
    }
    rows = []
    for c in df.columns:
        s = df[c]
        r = {"sheet": sheet, "column": c, "missing": int(s.isna().sum()), "missing_pct": round(100 * s.isna().mean(), 2),
             "pandas_inferred_dtype": str(s.infer_objects().dtype),
             "python_types": ", ".join(f"{k}:{v}" for k, v in s.map(type_name).value_counts().items()),
             "n_unique": int(s.astype(str).where(s.notna()).nunique())}
        if c in NUMERIC:
            x = pd.to_numeric(s, errors="coerce")
            r.update(numeric_min=x.min(), numeric_max=x.max(), numeric_median=x.median(),
                     non_numeric_non_null=int((x.isna() & s.notna()).sum()))
        elif r["n_unique"] <= 30:
            r["top_values"] = "; ".join(f"{k} ({v})" for k, v in s.value_counts().head(8).items())
        else:
            r["top_values"] = "; ".join(f"{k} ({v})" for k, v in s.value_counts().head(3).items())
        rows.append(r)
    return summary, pd.DataFrame(rows)

s_pre, p_pre = profile(pre, PRE)
s_post, p_post = profile(post, POST)
sheet_summary = pd.DataFrame([s_pre, s_post]).set_index("sheet").T
sheet_summary.to_csv(TABLES / "sheet_summary.csv")
column_profile = pd.concat([p_pre, p_post])
column_profile.to_csv(TABLES / "column_profile.csv", index=False)
sheet_summary

sheet,Dataset (up to Q1 2016),Dataset (from Q2 2016)
rows,49491,204469
columns_read,41,47
meaningful_columns,41,42
first_period,2011_1Q,2016_2Q
last_period,2016_1Q,2025_3Q
n_periods,17,37
unique_source_countries,35,49
unique_destination_countries,105,105
unique_corridors,310,372
unique_source_dest_pairs,310,368


In [20]:
column_profile[column_profile.sheet == PRE].drop(columns="sheet")

,column,missing,missing_pct,pandas_inferred_dtype,python_types,n_unique,numeric_min,numeric_max,numeric_median,non_numeric_non_null,top_values
0,id,0,0.00,int64,int:49491,49486,1.00000,59392.000,34652.00,0.0,NaN
1,period,0,0.00,object,str:49491,17,NaN,NaN,NaN,NaN,2016_1Q (3410); 2015_4Q (3397); 2014_1Q (2983); 2014_2Q (2951); 2013_2Q (2948); 2013_3Q (2941); 2013_1Q (2924); 2012...
2,source_code,0,0.00,object,str:49491,35,NaN,NaN,NaN,NaN,USA (6960); GBR (6405); AUS (3998)
3,source_name,0,0.00,object,str:49491,35,NaN,NaN,NaN,NaN,United States (6960); United Kingdom (6405); Australia (3998)
4,source_region,0,0.00,object,str:49491,4,NaN,NaN,NaN,NaN,.. (45452); Sub-Saharan Africa (2374); East Asia & Pacific (896); Latin America & Caribbean (769)
5,source_income,0,0.00,object,str:49491,5,NaN,NaN,NaN,NaN,High income: OECD (38722); High income: nonOECD (6454); Upper middle income (3224); Low income (978); Lower middle i...
6,source_lending,0,0.00,object,str:49491,3,NaN,NaN,NaN,NaN,.. (42712); IBRD (5688); IDA (1091)
7,source_G8G20,0,0.00,object,str:49491,3,NaN,NaN,NaN,NaN,G8/G20 (28373); .. (13748); G20 (7370)
8,destination_code,0,0.00,object,str:49491,105,NaN,NaN,NaN,NaN,PHL (5145); IND (3735); CHN (2824)
9,destination_name,0,0.00,object,str:49491,105,NaN,NaN,NaN,NaN,Philippines (5145); India (3735); China (2824)


In [21]:
column_profile[column_profile.sheet == POST].drop(columns="sheet")

,column,missing,missing_pct,pandas_inferred_dtype,python_types,n_unique,numeric_min,numeric_max,numeric_median,non_numeric_non_null,top_values
0,id,0,0.00,object,"int:191181, str:13288",204469,3.240001e+06,1.220195e+09,6.251916e+08,0.0,NaN
1,period,0,0.00,object,str:204469,37,NaN,NaN,NaN,NaN,2022_4Q (7608); 2023_4Q (7588); 2023_1Q (7582)
2,source_code,0,0.00,object,str:204469,49,NaN,NaN,NaN,NaN,USA (29542); GBR (23857); ITA (15334)
3,source_name,0,0.00,object,str:204469,51,NaN,NaN,NaN,NaN,United States (29542); United Kingdom (23857); Italy (15334)
4,source_region,0,0.00,object,str:204469,8,NaN,NaN,NaN,NaN,.. (178796); Sub-Saharan Africa (11183); East Asia & Pacific (9774); South Asia (1476); Latin America & Caribbean (1...
5,source_income,0,0.00,object,str:204469,5,NaN,NaN,NaN,NaN,High income (177359); Upper middle income (20484); Lower middle income (5101); Low income (1507); Lower income (18)
6,source_lending,0,0.00,object,str:204469,4,NaN,NaN,NaN,NaN,.. (175978); IBRD (23877); Blend (2792); IDA (1822)
7,source_G8G20,0,0.00,object,str:204469,3,NaN,NaN,NaN,NaN,G8/G20 (106770); .. (71450); G20 (26249)
8,destination_code,0,0.00,object,str:204469,105,NaN,NaN,NaN,NaN,IND (15757); PHL (14613); VNM (8936)
9,destination_name,0,0.00,object,str:204469,108,NaN,NaN,NaN,NaN,India (15757); Philippines (14613); Vietnam (8936)


In [22]:
period_counts = pd.concat([pre.period.value_counts(), post.period.value_counts()], axis=1, keys=["pre", "post"]).sort_index()
period_counts.to_csv(TABLES / "rows_per_period.csv")
all_q = [f"{y}_{q}Q" for y in range(2011, 2026) for q in range(1, 5)]
all_q = all_q[:all_q.index("2025_3Q") + 1]
print("Quarters with no rows in either sheet:", [q for q in all_q if q not in period_counts.index])
period_counts.T

Quarters with no rows in either sheet: ['2011_2Q', '2011_4Q', '2012_2Q', '2012_4Q', '2025_2Q']


period,2011_1Q,2011_3Q,2012_1Q,2012_3Q,2013_1Q,2013_2Q,2013_3Q,2013_4Q,2014_1Q,2014_2Q,2014_3Q,2014_4Q,2015_1Q,2015_2Q,2015_3Q,2015_4Q,2016_1Q,2016_2Q,2016_3Q,2016_4Q,2017_1Q,2017_2Q,2017_3Q,2017_4Q,2018_1Q,2018_2Q,2018_3Q,2018_4Q,2019_1Q,2019_2Q,2019_3Q,2019_4Q,2020_1Q,2020_2Q,2020_3Q,2020_4Q,2021_1Q,2021_2Q,2021_3Q,2021_4Q,2022_1Q,2022_2Q,2022_3Q,2022_4Q,2023_1Q,2023_2Q,2023_3Q,2023_4Q,2024_1Q,2024_2Q,2024_3Q,2024_4Q,2025_1Q,2025_3Q
pre,2603.0,2668.0,2638.0,2907.0,2924.0,2948.0,2941.0,2904.0,2983.0,2951.0,2890.0,2881.0,2821.0,2794.0,2831.0,3397.0,3410.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
post,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3905.0,3958.0,3932.0,3954.0,4093.0,4174.0,4185.0,4217.0,4270.0,4423.0,4637.0,4757.0,4952.0,5123.0,5176.0,5197.0,4878.0,4912.0,4897.0,4871.0,5507.0,5746.0,5986.0,6034.0,5892.0,5704.0,7608.0,7582.0,7581.0,7537.0,7588.0,7467.0,7318.0,6636.0,6655.0,6647.0,6470.0


## 11. Cost-field validation
The Legend defines (per surveyed amount `cc(X)`):

* `lcu fee` — fee in local currency;
* `lcu fx rate` — exchange rate applied by the RSP;
* `inter lcu bank fx` — interbank exchange rate;
* `fx margin` — percentage difference between the RSP rate and the interbank rate;
* `total cost %` — total cost of the transaction in percentage.

No explicit equation is given in the workbook. The natural reading — and the one tested here — is
`total cost % ≈ fee / lcu amount × 100 + fx margin` and `fx margin ≈ (1 − lcu fx rate / inter lcu bank fx) × 100`.
We **only measure** how well these hold; nothing is corrected.

In [23]:
num = lambda s: pd.to_numeric(s, errors="coerce")
cost_rows, examples = [], {}
for sheet, df in [(PRE, pre), (POST, post)]:
    inter = num(df["inter lcu bank fx"])
    transparent = df.transparent.astype(str).str.strip().str.lower()
    for k in ["cc1", "cc2"]:
        amt, fee, fx = num(df[f"{k} lcu amount"]), num(df[f"{k} lcu fee"]), num(df[f"{k} lcu fx rate"])
        mar, tot = num(df[f"{k} fx margin"]), num(df[f"{k} total cost %"])
        d_total = (tot - (fee / amt * 100 + mar)).abs()
        d_margin = (mar - (1 - fx / inter) * 100).abs()
        for label, d, denom in [("total = fee/amount*100 + margin", d_total, amt), ("margin = (1 - fx/inter)*100", d_margin, inter)]:
            zero_denom = int((denom == 0).sum())
            d = d.replace([np.inf, -np.inf], np.nan)
            ok = d.notna()
            cost_rows.append({"sheet": sheet, "amount": k, "relationship": label, "rows_testable": int(ok.sum()),
                              "rows_with_zero_denominator": zero_denom,
                              "within_0.01pp": round(100 * (d[ok] <= 0.01).mean(), 2),
                              "within_0.1pp": round(100 * (d[ok] <= 0.1).mean(), 2),
                              "within_1pp": round(100 * (d[ok] <= 1).mean(), 2),
                              "rows_off_by_more_than_1pp": int((d[ok] > 1).sum()),
                              "max_abs_diff_pp": round(d[ok].max(), 3)})
            examples[(sheet, k, label)] = df.loc[ok & (d > 1), ["period", "corridor", "firm", f"{k} lcu amount", f"{k} lcu fee",
                                                           f"{k} lcu fx rate", "inter lcu bank fx", f"{k} fx margin",
                                                           f"{k} total cost %", "transparent"]].head(5)
cost_checks = pd.DataFrame(cost_rows)
cost_checks.to_csv(TABLES / "cost_relationship_checks.csv", index=False)
cost_checks

,sheet,amount,relationship,rows_testable,rows_with_zero_denominator,within_0.01pp,within_0.1pp,within_1pp,rows_off_by_more_than_1pp,max_abs_diff_pp
0,Dataset (up to Q1 2016),cc1,total = fee/amount*100 + margin,49381,0,99.57,99.58,99.80,100,26.471
1,Dataset (up to Q1 2016),cc1,margin = (1 - fx/inter)*100,49490,0,96.02,97.93,99.91,43,95.855
2,Dataset (up to Q1 2016),cc2,total = fee/amount*100 + margin,49178,103,99.64,99.66,99.94,30,12.095
3,Dataset (up to Q1 2016),cc2,margin = (1 - fx/inter)*100,49191,0,95.99,97.91,99.91,45,6787.130
4,Dataset (from Q2 2016),cc1,total = fee/amount*100 + margin,204445,0,99.96,99.98,99.99,29,23.320
5,Dataset (from Q2 2016),cc1,margin = (1 - fx/inter)*100,204446,1,94.83,97.81,99.79,426,370469.639
6,Dataset (from Q2 2016),cc2,total = fee/amount*100 + margin,203396,0,99.97,99.98,99.99,25,7.143
7,Dataset (from Q2 2016),cc2,margin = (1 - fx/inter)*100,203395,1,95.16,98.04,99.84,326,370469.639


In [24]:
for key, ex in examples.items():
    if len(ex):
        print(key); print(ex.to_string()); print()

('Dataset (up to Q1 2016)', 'cc1', 'total = fee/amount*100 + margin')
       period corridor             firm  cc1 lcu amount  cc1 lcu fee  cc1 lcu fx rate  inter lcu bank fx  cc1 fx margin  cc1 total cost % transparent
597   2011_1Q   DEUHRV        MoneyGram           160.0         16.0         6.970000             7.4200           6.06              9.26         yes
928   2011_1Q   JPNPHL        SBI remit         17000.0        880.0         0.011975             0.0120           0.21              9.03         yes
936   2011_1Q   JPNPHL        SBI remit         17000.0        880.0         0.509707             0.5363           4.96             13.78         yes
3469  2011_3Q   JPNBRA  Banco do Brasil         17000.0       1500.0         0.020500             0.0214           4.21             11.26         yes
7881  2012_1Q   USAPHL        MoneyGram           200.0         16.0        41.650000            42.5300           2.07              7.06         yes

('Dataset (up to Q1 2016)', '

In [25]:
print("Surveyed amounts (denomination, USD) and relationship between cc1 and cc2:")
for sheet, df in [(PRE, pre), (POST, post)]:
    d1, d2 = num(df["cc1 denomination amount"]), num(df["cc2 denomination amount"])
    print(f"  {sheet}: cc1 denomination {d1.value_counts().to_dict()}  cc2 denomination {d2.value_counts().to_dict()}")
    ratio = num(df["cc2 lcu amount"]) / num(df["cc1 lcu amount"])
    print(f"    cc2/cc1 lcu amount ratio: median {ratio.median():.3f}, share within 2.5±0.01: {((ratio-2.5).abs()<=0.01).mean():.4f}")
    print(f"    cc1 lcu code == cc2 lcu code: {(df['cc1 lcu code'] == df['cc2 lcu code']).mean():.4f}")
    print(f"    cc1 fx rate == cc2 fx rate: {(num(df['cc1 lcu fx rate']) == num(df['cc2 lcu fx rate'])).mean():.4f}")

Surveyed amounts (denomination, USD) and relationship between cc1 and cc2:
  Dataset (up to Q1 2016): cc1 denomination {200.0: 49406, 201.0: 3, 6.0: 1, 202.0: 1}  cc2 denomination {500.0: 49401, 501.0: 9, 502.0: 3}
    cc2/cc1 lcu amount ratio: median 2.499, share within 2.5±0.01: 0.5950
    cc1 lcu code == cc2 lcu code: 0.9991
    cc1 fx rate == cc2 fx rate: 0.9731
  Dataset (from Q2 2016): cc1 denomination {200.0: 204469}  cc2 denomination {500.0: 204469}
    cc2/cc1 lcu amount ratio: median 2.500, share within 2.5±0.01: 0.5644
    cc1 lcu code == cc2 lcu code: 1.0000
    cc1 fx rate == cc2 fx rate: 0.9630


### 11.1 Transparency and zero FX margins

In [26]:
tr_rows = []
for sheet, df in [(PRE, pre), (POST, post)]:
    transparent = df.transparent.astype(str).str.strip().str.lower()
    note_col = "note1" if "note1" in df else "Standard Note"
    note_nt = df[note_col].astype(str).str.contains("not transparent", case=False, na=False)
    for k in ["cc1", "cc2"]:
        mar, fx, inter = num(df[f"{k} fx margin"]), num(df[f"{k} lcu fx rate"]), num(df["inter lcu bank fx"])
        for t in sorted(transparent.unique()):
            m = transparent == t
            tr_rows.append({"sheet": sheet, "amount": k, "transparent": t, "rows": int(m.sum()),
                            "margin_eq_0": int((m & (mar == 0)).sum()),
                            "margin_lt_0": int((m & (mar < 0)).sum()),
                            "fx_rate_eq_inter_rate": int((m & (fx == inter)).sum()),
                            "fx_rate_eq_1": int((m & (fx == 1)).sum()),
                            "note_says_not_transparent": int((m & note_nt).sum())})
transparency = pd.DataFrame(tr_rows)
transparency.to_csv(TABLES / "transparency_margin_checks.csv", index=False)
transparency

,sheet,amount,transparent,rows,margin_eq_0,margin_lt_0,fx_rate_eq_inter_rate,fx_rate_eq_1,note_says_not_transparent
0,Dataset (up to Q1 2016),cc1,no,4057,4036,1,4037,4033,3962
1,Dataset (up to Q1 2016),cc1,yes,45434,8866,1800,8507,6782,6
2,Dataset (up to Q1 2016),cc2,no,4057,3983,2,3990,3986,3962
3,Dataset (up to Q1 2016),cc2,yes,45434,8817,1806,8557,6733,6
4,Dataset (from Q2 2016),cc1,no,3361,3351,0,3352,3352,3315
5,Dataset (from Q2 2016),cc1,yes,201108,25885,9169,25261,21169,282
6,Dataset (from Q2 2016),cc2,no,3361,3349,0,3350,3350,3315
7,Dataset (from Q2 2016),cc2,yes,201108,25739,9129,25121,21037,282


In [27]:
print("Periods containing transparent = 'no':")
for sheet, df in [(PRE, pre), (POST, post)]:
    t = df.transparent.astype(str).str.lower()
    print(" ", sheet, df.loc[t == "no", "period"].min(), "->", df.loc[t == "no", "period"].max(), int((t == "no").sum()), "rows")
print("\nNote texts attached to non-transparent rows (post):")
print(post.loc[post.transparent.astype(str).str.lower() == "no", "Standard Note"].value_counts(dropna=False).head(5).to_string())
print("\nNegative-margin / negative-total rows:")
for sheet, df in [(PRE, pre), (POST, post)]:
    for k in ["cc1", "cc2"]:
        mar, tot = num(df[f"{k} fx margin"]), num(df[f"{k} total cost %"])
        print(f"  {sheet[:24]:24s} {k}: margin<0 {int((mar<0).sum()):5d}  total<0 {int((tot<0).sum()):5d}  total==0 {int((tot==0).sum()):4d}")

Periods containing transparent = 'no':
  Dataset (up to Q1 2016) 2011_1Q -> 2016_1Q 4057 rows
  Dataset (from Q2 2016) 2016_2Q -> 2021_4Q 3361 rows

Note texts attached to non-transparent rows (post):
Standard Note
This RSP is not transparent: the exchange rate applied to the transaction was not disclosed. The 0% in the exchange rate margin does NOT necessarily mean that there is no exchange rate cost, but rather that this cost is not disclosed to the sender at the time of sending.    2993
This RSP is not transparent. In the receiving country, RSPs are not allowed to pay out transactions in any currency other than the legal tender and an exchange rate to the receiving country's local currency was not provided.                                                    120
This RSP is not transparent: the exchange rate applied to the transaction was not disclosed. The 0% in the exchange rate margin does NOT necessarily mean that there is no exchange rate cost, but rather that this cost is not d

## 12. Natural unit of observation — candidate keys
The Legend says `id` is the "id number applied to each service", and each row carries costs for two surveyed
amounts (`cc1`, `cc2`) side by side. So a row is a **service observation in a period** with two amount
scenarios in wide form. We test which combinations of descriptive fields uniquely identify rows.

In [28]:
def key_test(df, cols):
    k = df[cols].astype(str)
    dup_mask = k.duplicated(keep=False)
    return {"columns": " + ".join(cols), "n_cols": len(cols), "rows": len(df), "distinct_keys": int(k.drop_duplicates().shape[0]),
            "rows_in_duplicate_groups": int(dup_mask.sum()), "unique": bool(not dup_mask.any())}

BASE = ["period", "source_code", "destination_code", "firm"]
cand_pre = [["id"], ["period", "id"], BASE, BASE + ["product"], BASE + ["product", "sending location"],
            BASE + ["product", "sending location", "speed actual"],
            BASE + ["product", "sending location", "speed actual", "pick-up method", "coverage"],
            BASE + ["product", "sending location", "speed actual", "pick-up method", "coverage", "cc1 lcu fee", "cc1 total cost %"],
            [c for c in pre.columns if c != "id"]]
cand_post = [["id"], ["period", "id"], BASE, BASE + ["payment instrument"], BASE + ["payment instrument", "access point"],
             BASE + ["payment instrument", "access point", "pickup method"],
             BASE + ["payment instrument", "access point", "pickup method", "speed actual"],
             BASE + ["payment instrument", "access point", "pickup method", "speed actual", "pickup location", "receiving network coverage"],
             BASE + ["payment instrument", "access point", "pickup method", "speed actual", "pickup location", "receiving network coverage",
                     "cc1 lcu fee", "cc1 total cost %"],
             [c for c in post.columns if c != "id"]]
keys = pd.DataFrame([dict(sheet=PRE, **key_test(pre, c)) for c in cand_pre] +
                    [dict(sheet=POST, **key_test(post, c)) for c in cand_post])
keys.loc[keys.n_cols > 20, "columns"] = "ALL columns except id"
keys.to_csv(TABLES / "candidate_key_tests.csv", index=False)
keys

,sheet,columns,n_cols,rows,distinct_keys,rows_in_duplicate_groups,unique
0,Dataset (up to Q1 2016),id,1,49491,49486,10,False
1,Dataset (up to Q1 2016),period + id,2,49491,49491,0,True
2,Dataset (up to Q1 2016),period + source_code + destination_code + firm,4,49491,34386,25225,False
3,Dataset (up to Q1 2016),period + source_code + destination_code + firm + product,5,49491,43836,10634,False
4,Dataset (up to Q1 2016),period + source_code + destination_code + firm + product + sending location,6,49491,44926,8475,False
5,Dataset (up to Q1 2016),period + source_code + destination_code + firm + product + sending location + speed actual,7,49491,47011,4591,False
6,Dataset (up to Q1 2016),period + source_code + destination_code + firm + product + sending location + speed actual + pick-up method + coverage,9,49491,47304,4016,False
7,Dataset (up to Q1 2016),period + source_code + destination_code + firm + product + sending location + speed actual + pick-up method + covera...,11,49491,49296,390,False
8,Dataset (up to Q1 2016),ALL columns except id,40,49491,49358,266,False
9,Dataset (from Q2 2016),id,1,204469,204469,0,True


In [29]:
dup_ids = pre[pre.id.astype(str).duplicated(keep=False)].sort_values("id")
print("Pre-2016 rows sharing an id:", len(dup_ids))
print(dup_ids[["id", "period", "corridor", "firm", "product"]].head(12).to_string())
print("\nid overlap between sheets:", len(set(pre.id.astype(str)) & set(post.id.astype(str))))
print("Does an id persist across periods in post? rows per id max:", post.id.astype(str).value_counts().max())
ex = post[post.drop(columns="id").astype(str).duplicated(keep=False)].sort_values(["period", "corridor", "firm"])
print("\nExample post rows identical except for id:")
print(ex[["id", "period", "corridor", "firm", "payment instrument", "access point", "cc1 total cost %"]].head(6).to_string())

Pre-2016 rows sharing an id: 10
          id   period corridor                                 firm                                 product
33265  44140  2014_4Q   ZAFZWE                        Standard Bank                      Account to account
36206  44140  2015_1Q   AREEGY                   Al Fardan Exchange                      Account to account
34521  44141  2015_1Q   BELCOD                        Western Union  Cash to cash,Credit/debit card service
33262  44141  2014_4Q   ZAFZWE                              Nedbank                      Account to account
33266  44142  2014_4Q   ZAFZWE  First National Bank of South Africa                      Account to account
36212  44142  2015_1Q   AREEGY                         UAE Exchange                      Account to account
34522  44143  2015_1Q   BELCOD                            MoneyGram                            Cash to cash
33260  44143  2014_4Q   ZAFZWE                                 ABSA                      Account to acco


Example post rows identical except for id:
             id   period corridor        firm     payment instrument                      access point cc1 total cost %
169   620160170  2016_2Q   AUSIDN   MoneyGram                   Cash                             Agent            10.56
170   620160171  2016_2Q   AUSIDN   MoneyGram                   Cash                             Agent            10.56
2385  620162449  2016_2Q   JPNBRA     GoRemit  Bank account transfer        Agent,Call Center,Internet            11.76
2386  620162450  2016_2Q   JPNBRA     GoRemit  Bank account transfer        Agent,Call Center,Internet            11.76
2274  620162338  2016_2Q   JPNBRA  Seven Bank  Bank account transfer  Bank branch,Call Center,Internet            10.93
2384  620162448  2016_2Q   JPNBRA  Seven Bank  Bank account transfer  Bank branch,Call Center,Internet            10.93


Result: `id` is unique within the post-2016 sheet, and unique within the pre-2016 sheet except for a small number
of reused ids across periods; `period + id` is unique in both. `id` does **not** persist across quarters, so it is a
row identifier, not a stable service identifier. No combination of *descriptive* fields is unique: even all
columns except `id` leave exact duplicate rows. Descriptive fields therefore only approximately identify a
service; longitudinal tracking of "the same service" across quarters would need an assumption-based key.

## 13. Cross-sheet overlap (for harmonization feasibility only)

In [30]:
overlap = {
    "firms_in_both": len(set(pre.firm) & set(post.firm)),
    "firms_pre_only": len(set(pre.firm) - set(post.firm)),
    "firms_post_only": len(set(post.firm) - set(pre.firm)),
    "corridors_in_both": len(set(pre.corridor) & set(post.corridor)),
    "corridors_pre_only": len(set(pre.corridor) - set(post.corridor)),
    "corridors_post_only": len(set(post.corridor) - set(pre.corridor)),
    "income_labels_pre": sorted(pre.source_income.dropna().unique().tolist() + pre.destination_income.dropna().unique().tolist()),
    "income_labels_post": sorted(set(post.source_income.dropna()) | set(post.destination_income.dropna())),
    "region_labels_pre": sorted(set(pre.source_region.dropna()) | set(pre.destination_region.dropna())),
    "region_labels_post": sorted(set(post.source_region.dropna()) | set(post.destination_region.dropna())),
}
for k, v in overlap.items():
    print(f"{k}: {sorted(set(v)) if isinstance(v, list) else v}")

firms_in_both: 440
firms_pre_only: 197
firms_post_only: 273
corridors_in_both: 300
corridors_pre_only: 10
corridors_post_only: 72
income_labels_pre: ['..', 'High income: OECD', 'High income: nonOECD', 'Low income', 'Lower middle income', 'Upper middle income']
income_labels_post: ['High income', 'Low income', 'Lower income', 'Lower middle income', 'Upper middle income']
region_labels_pre: ['..', 'East Asia & Pacific', 'Europe & Central Asia', 'Latin America & Caribbean', 'Middle East & North Africa', 'South Asia', 'Sub-Saharan Africa']
region_labels_post: ['..', 'East Asia & Pacific', 'Europe & Central Asia', 'Latin America & Caribbean', 'Middle East & North Africa', 'Middle East, North Africa, Afghanistan & Pakistan', 'South Asia', 'Sub-Saharan Africa']


## 14. Final integrity check
Confirm the raw workbook is byte-for-byte unchanged after the run.

In [31]:
zf.close()
hash_after = sha256(RAW)
print("sha256 after run:", hash_after)
assert hash_after == hash_before == EXPECTED_SHA256
print("Raw workbook unchanged.")
print("Tables written:", sorted(p.name for p in TABLES.glob("*.csv")))

sha256 after run: f1d7265b1856d61812c158d3d03fae686dbff58735deabe016a6bfe23482d2a8
Raw workbook unchanged.
Tables written: ['candidate_key_tests.csv', 'column_profile.csv', 'cost_relationship_checks.csv', 'label_variants.csv', 'legend_raw.csv', 'mixed_value_types.csv', 'placeholder_counts.csv', 'post2016_xml_column_scan.csv', 'rows_per_period.csv', 'schema_header_comparison.csv', 'sheet_summary.csv', 'transparency_margin_checks.csv']
